# Customer Segmentation Using K-Means Clustering
### Unsupervised Machine Learning | Business Analytics
---

## 1. Imports & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 150,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'figure.titlesize': 16,
})

PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B2',
           '#937860', '#DA8BC3', '#8C8C8C', '#CCB974', '#64B5CD']
SEED = 42

## 2. Data Loading

In [ ]:
df_raw = pd.read_csv('marketing_campaign.csv', sep='\t')

print(f"Dataset Shape  : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"Memory Usage   : {df_raw.memory_usage(deep=True).sum() / 1024:.1f} KB")
print("\nColumn Overview:")
print(df_raw.dtypes.to_frame('dtype').join(df_raw.isnull().sum().to_frame('nulls')))

In [ ]:
df_raw.head()

In [ ]:
df_raw.describe(include='all').T

## 3. Data Preprocessing

In [ ]:
df = df_raw.copy()

df['Income'] = df['Income'].fillna(df['Income'].median())

df['Age'] = 2024 - df['Year_Birth']
df['TotalSpending'] = (df['MntWines'] + df['MntFruits'] + df['MntMeatProducts'] +
                       df['MntFishProducts'] + df['MntSweetProducts'] + df['MntGoldProds'])
df['TotalPurchases'] = (df['NumWebPurchases'] + df['NumCatalogPurchases'] +
                        df['NumStorePurchases'] + df['NumDealsPurchases'])
df['Children'] = df['Kidhome'] + df['Teenhome']

age_cap = df['Age'].quantile(0.99)
income_cap = df['Income'].quantile(0.99)
df = df[df['Age'] <= age_cap].copy()
df = df[df['Income'] <= income_cap].copy()

print(f"Cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Remaining nulls: {df.isnull().sum().sum()}")

features = ['Income', 'TotalSpending', 'Age', 'TotalPurchases', 'Children', 'Recency']
X = df[features].copy()
print(f"\nFeature matrix shape: {X.shape}")
X.describe().round(2)

In [ ]:
scaler_std = StandardScaler()
X_std = pd.DataFrame(scaler_std.fit_transform(X), columns=features)

scaler_mm = MinMaxScaler()
X_mm = pd.DataFrame(scaler_mm.fit_transform(X), columns=features)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Feature Scaling Comparison', fontweight='bold', y=1.02)

axes[0].boxplot([X[f] for f in features], labels=features, patch_artist=True,
                boxprops=dict(facecolor='#4C72B0', alpha=0.7))
axes[0].set_title('Original (Unscaled)')
axes[0].tick_params(axis='x', rotation=25)

axes[1].boxplot([X_std[f] for f in features], labels=features, patch_artist=True,
                boxprops=dict(facecolor='#DD8452', alpha=0.7))
axes[1].set_title('StandardScaler (Z-score)')
axes[1].tick_params(axis='x', rotation=25)

axes[2].boxplot([X_mm[f] for f in features], labels=features, patch_artist=True,
                boxprops=dict(facecolor='#55A868', alpha=0.7))
axes[2].set_title('MinMaxScaler (0–1)')
axes[2].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig('scaling_comparison.png', bbox_inches='tight')
plt.show()

X_scaled = X_std.values
print("Using StandardScaler for clustering (recommended for K-Means)")

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Feature Distributions', fontweight='bold', fontsize=16)

colors = PALETTE[:6]
for ax, feat, col in zip(axes.flat, features, colors):
    ax.hist(X[feat], bins=40, color=col, alpha=0.8, edgecolor='white', linewidth=0.5)
    ax.axvline(X[feat].mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean: {X[feat].mean():.0f}')
    ax.axvline(X[feat].median(), color='red', linestyle=':', linewidth=1.5, label=f'Median: {X[feat].median():.0f}')
    ax.set_title(feat)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
corr = X.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Key Feature Relationships', fontweight='bold', fontsize=16)

pairs = [('Income', 'TotalSpending'), ('Age', 'TotalSpending'), ('Income', 'TotalPurchases')]
for ax, (xf, yf) in zip(axes, pairs):
    ax.scatter(X[xf], X[yf], alpha=0.35, s=18, color='#4C72B0', edgecolors='none')
    ax.set_xlabel(xf)
    ax.set_ylabel(yf)
    ax.set_title(f'{xf} vs {yf}')
    z = np.polyfit(X[xf], X[yf], 1)
    p = np.poly1d(z)
    xline = np.linspace(X[xf].min(), X[xf].max(), 200)
    ax.plot(xline, p(xline), 'r--', linewidth=1.5, alpha=0.8)

plt.tight_layout()
plt.savefig('feature_relationships.png', bbox_inches='tight')
plt.show()

## 5. Optimal K — Elbow Method

In [ ]:
wcss = []
K_range = range(1, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=SEED)
    km.fit(X_scaled)
    wcss.append(km.inertia_)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(K_range, wcss, marker='o', markersize=9, color='#4C72B0',
        linewidth=2.5, markerfacecolor='white', markeredgewidth=2.5)

deltas = np.diff(wcss)
delta2 = np.diff(deltas)
elbow_k = int(np.argmax(np.abs(delta2)) + 2)

ax.axvline(elbow_k, color='#C44E52', linestyle='--', linewidth=2,
           label=f'Elbow at K = {elbow_k}')
ax.fill_betweenx([min(wcss)*0.97, max(wcss)*1.01],
                 elbow_k - 0.35, elbow_k + 0.35,
                 alpha=0.12, color='#C44E52')

ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('WCSS (Within-Cluster Sum of Squares)')
ax.set_title('Elbow Method — Optimal K Selection', fontweight='bold')
ax.set_xticks(list(K_range))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('elbow_method.png', bbox_inches='tight')
plt.show()

print(f"Elbow detected at K = {elbow_k}")
for k, w in zip(K_range, wcss):
    print(f"  K={k:2d}  WCSS = {w:,.0f}")

## 6. Silhouette Score Analysis

In [ ]:
sil_scores = []
K_sil = range(2, 11)

for k in K_sil:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=SEED)
    labels = km.fit_predict(X_scaled)
    sil_scores.append(silhouette_score(X_scaled, labels))

best_sil_k = int(K_sil[int(np.argmax(sil_scores))])
best_sil_score = max(sil_scores)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(K_sil, sil_scores, color=['#C44E52' if k == best_sil_k else '#4C72B0' for k in K_sil],
              alpha=0.85, edgecolor='white', linewidth=0.8, width=0.65)
ax.plot(K_sil, sil_scores, marker='D', color='black', linewidth=1.5,
        markersize=6, markerfacecolor='white', markeredgewidth=1.5, zorder=5)
ax.set_xlabel('Number of Clusters (K)')
ax.set_ylabel('Silhouette Score')
ax.set_title('Silhouette Score per K — Cluster Separation Quality', fontweight='bold')
ax.set_xticks(list(K_sil))
ax.axhline(best_sil_score, color='#DD8452', linestyle=':', linewidth=1.5, alpha=0.6)

for bar, score in zip(bars, sil_scores):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
            f'{score:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('silhouette_scores.png', bbox_inches='tight')
plt.show()

print(f"Best Silhouette Score = {best_sil_score:.4f} at K = {best_sil_k}")
for k, s in zip(K_sil, sil_scores):
    marker = '  ← best' if k == best_sil_k else ''
    print(f"  K={k:2d}  Silhouette = {s:.4f}{marker}")

## 7. K-Means Clustering

In [ ]:
OPTIMAL_K = 4

kmeans = KMeans(n_clusters=OPTIMAL_K, init='k-means++', n_init=20,
                max_iter=500, random_state=SEED)
df['Cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means trained with K = {OPTIMAL_K}")
print(f"Inertia (WCSS): {kmeans.inertia_:,.2f}")
print(f"Iterations to converge: {kmeans.n_iter_}")
print("\nCluster sizes:")
print(df['Cluster'].value_counts().sort_index().to_frame('Count')
      .assign(Pct=lambda x: (x['Count'] / len(df) * 100).round(1).astype(str) + '%'))

## 8. Cluster Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Customer Clusters — Feature Space', fontweight='bold', fontsize=16)

cluster_colors = {0: '#4C72B0', 1: '#DD8452', 2: '#55A868', 3: '#C44E52'}

for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    axes[0].scatter(df.loc[mask, 'Income'], df.loc[mask, 'TotalSpending'],
                    alpha=0.55, s=40, color=cluster_colors[cluster_id],
                    label=f'Cluster {cluster_id}', edgecolors='none')

axes[0].set_xlabel('Annual Income ($)')
axes[0].set_ylabel('Total Spending ($)')
axes[0].set_title('Income vs Total Spending')
axes[0].legend(markerscale=1.5)
axes[0].xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    axes[1].scatter(df.loc[mask, 'Age'], df.loc[mask, 'TotalSpending'],
                    alpha=0.55, s=40, color=cluster_colors[cluster_id],
                    label=f'Cluster {cluster_id}', edgecolors='none')

axes[1].set_xlabel('Age (years)')
axes[1].set_ylabel('Total Spending ($)')
axes[1].set_title('Age vs Total Spending')
axes[1].legend(markerscale=1.5)

plt.tight_layout()
plt.savefig('cluster_scatter.png', bbox_inches='tight')
plt.show()

## 9. Cluster Summary Table

In [ ]:
summary = df.groupby('Cluster').agg(
    Avg_Income=('Income', 'mean'),
    Avg_TotalSpending=('TotalSpending', 'mean'),
    Avg_Age=('Age', 'mean'),
    Avg_Purchases=('TotalPurchases', 'mean'),
    Avg_Children=('Children', 'mean'),
    Count=('Cluster', 'count')
).round(2)

summary['Pct_of_Customers'] = (summary['Count'] / len(df) * 100).round(1).astype(str) + '%'
summary['Avg_Income'] = summary['Avg_Income'].apply(lambda x: f'${x:,.0f}')
summary['Avg_TotalSpending'] = summary['Avg_TotalSpending'].apply(lambda x: f'${x:,.0f}')
summary['Avg_Age'] = summary['Avg_Age'].apply(lambda x: f'{x:.0f}')
summary['Avg_Purchases'] = summary['Avg_Purchases'].apply(lambda x: f'{x:.1f}')

print("=" * 70)
print("CLUSTER SUMMARY TABLE")
print("=" * 70)
print(summary.to_string())
print("=" * 70)
summary

## 10. Cluster Profiling — Business Names

In [ ]:
numeric_summary = df.groupby('Cluster').agg(
    Avg_Income=('Income', 'mean'),
    Avg_Spending=('TotalSpending', 'mean'),
    Avg_Age=('Age', 'mean'),
    Count=('Cluster', 'count')
)

income_median = numeric_summary['Avg_Income'].median()
spending_median = numeric_summary['Avg_Spending'].median()

def assign_business_name(row):
    high_income = row['Avg_Income'] >= income_median
    high_spending = row['Avg_Spending'] >= spending_median
    if high_income and high_spending:
        return 'Luxury Shoppers'
    elif high_income and not high_spending:
        return 'Potential Customers'
    elif not high_income and high_spending:
        return 'Value Seekers'
    else:
        return 'Budget Customers'

numeric_summary['Business_Name'] = numeric_summary.apply(assign_business_name, axis=1)
cluster_name_map = numeric_summary['Business_Name'].to_dict()

df['Segment'] = df['Cluster'].map(cluster_name_map)

print("Cluster → Business Name Mapping:")
for c, name in cluster_name_map.items():
    row = numeric_summary.loc[c]
    print(f"  Cluster {c} → {name:25s}  Income=${row['Avg_Income']:,.0f}  Spending=${row['Avg_Spending']:,.0f}")

print("\nSegment Distribution:")
print(df['Segment'].value_counts().to_frame('Count')
      .assign(Pct=(df['Segment'].value_counts() / len(df) * 100).round(1).astype(str) + '%'))

In [ ]:
SEGMENT_COLORS = {
    'Luxury Shoppers': '#C44E52',
    'Potential Customers': '#4C72B0',
    'Value Seekers': '#55A868',
    'Budget Customers': '#DD8452',
}

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('Customer Segments — Business View', fontweight='bold', fontsize=16)

for seg in df['Segment'].unique():
    mask = df['Segment'] == seg
    axes[0].scatter(df.loc[mask, 'Income'], df.loc[mask, 'TotalSpending'],
                    alpha=0.6, s=45, color=SEGMENT_COLORS.get(seg, 'grey'),
                    label=seg, edgecolors='none')

axes[0].set_xlabel('Annual Income ($)')
axes[0].set_ylabel('Total Spending ($)')
axes[0].set_title('Income vs Total Spending by Segment')
axes[0].legend(markerscale=1.5, loc='upper left')
axes[0].xaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

counts = df['Segment'].value_counts()
wedge_colors = [SEGMENT_COLORS.get(s, 'grey') for s in counts.index]
wedges, texts, autotexts = axes[1].pie(
    counts.values, labels=counts.index, autopct='%1.1f%%',
    colors=wedge_colors, startangle=140,
    wedgeprops=dict(linewidth=2, edgecolor='white'),
    textprops=dict(fontsize=10))
for at in autotexts:
    at.set_fontsize(9)
    at.set_fontweight('bold')
axes[1].set_title('Customer Segment Distribution')

plt.tight_layout()
plt.savefig('segment_profiles.png', bbox_inches='tight')
plt.show()

## 11. PCA Visualization

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

explained = pca.explained_variance_ratio_
print(f"PCA Explained Variance:")
print(f"  PC1 → {explained[0]*100:.1f}%")
print(f"  PC2 → {explained[1]*100:.1f}%")
print(f"  Total → {sum(explained)*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.suptitle('PCA — 2D Customer Cluster Visualization', fontweight='bold', fontsize=16)

for cluster_id in sorted(df['Cluster'].unique()):
    mask = df['Cluster'] == cluster_id
    seg_name = cluster_name_map[cluster_id]
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    alpha=0.55, s=30, label=f'C{cluster_id}: {seg_name}',
                    color=list(cluster_colors.values())[cluster_id], edgecolors='none')

centers_pca = pca.transform(kmeans.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1],
                marker='*', s=350, color='black', zorder=10, label='Centroids')
axes[0].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[0].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[0].set_title('Clusters in PCA Space')
axes[0].legend(markerscale=1.2, fontsize=9)

for seg in df['Segment'].unique():
    mask = df['Segment'] == seg
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1],
                    alpha=0.55, s=30, label=seg,
                    color=SEGMENT_COLORS.get(seg, 'grey'), edgecolors='none')
axes[1].scatter(centers_pca[:, 0], centers_pca[:, 1],
                marker='*', s=350, color='black', zorder=10, label='Centroids')
axes[1].set_xlabel(f'PC1 ({explained[0]*100:.1f}% variance)')
axes[1].set_ylabel(f'PC2 ({explained[1]*100:.1f}% variance)')
axes[1].set_title('Segments in PCA Space')
axes[1].legend(markerscale=1.2, fontsize=9)

for i, (x, y) in enumerate(centers_pca):
    for ax in axes:
        ax.annotate(f'C{i}', (x, y), textcoords='offset points', xytext=(8, 8),
                    fontsize=9, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.7))

plt.tight_layout()
plt.savefig('pca_visualization.png', bbox_inches='tight')
plt.show()

In [ ]:
pca_full = PCA(n_components=min(len(features), len(features)), random_state=SEED)
pca_full.fit(X_scaled)

cumulative_var = np.cumsum(pca_full.explained_variance_ratio_)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(range(1, len(features)+1), pca_full.explained_variance_ratio_ * 100,
       color='#4C72B0', alpha=0.8, label='Individual')
ax.plot(range(1, len(features)+1), cumulative_var * 100,
        marker='o', color='#C44E52', linewidth=2.5, markersize=7, label='Cumulative')
ax.axhline(80, color='grey', linestyle='--', linewidth=1, alpha=0.7, label='80% threshold')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained Variance (%)')
ax.set_title('PCA — Variance Explained per Component', fontweight='bold')
ax.set_xticks(range(1, len(features)+1))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance.png', bbox_inches='tight')
plt.show()

## 12. Cluster Recommendation System

In [ ]:
RECOMMENDATIONS = {
    'Luxury Shoppers': {
        'emoji': '💎',
        'products': ['Premium & exclusive product lines', 'Luxury brand collaborations',
                     'High-end subscription boxes', 'Personalised concierge services'],
        'campaigns': ['VIP loyalty rewards program', 'Early-access new collection emails',
                      'Invite-only private events', 'Personal shopper consultations'],
        'channels': ['Email (personalised)', 'Direct mail', 'Mobile app push (premium)'],
        'priority': 'HIGH — Highest CLV segment'
    },
    'Potential Customers': {
        'emoji': '🚀',
        'products': ['Mid-to-premium product bundles', 'Value-added service packages',
                     'Curated upgrade recommendations', 'Smart savings bundles'],
        'campaigns': ['Spending incentive challenges', 'Targeted upsell journeys',
                      'Seasonal discount events', '"Why not treat yourself?" messaging'],
        'channels': ['Retargeting ads', 'Email drip sequences', 'In-app notifications'],
        'priority': 'HIGH — Untapped revenue potential'
    },
    'Value Seekers': {
        'emoji': '🌟',
        'products': ['Quality mid-tier products', 'Category bundles at value price',
                     'Bestseller collections', 'Membership discounts'],
        'campaigns': ['Flash sales & weekend deals', 'Cashback loyalty schemes',
                      'Refer-a-friend programs', 'Points reward system'],
        'channels': ['Social media ads', 'SMS / WhatsApp', 'Email newsletters'],
        'priority': 'MEDIUM — Retention & spend growth'
    },
    'Budget Customers': {
        'emoji': '💡',
        'products': ['Entry-level & essentials range', 'Clearance & outlet offers',
                     'BOGO (Buy One Get One) deals', 'Subscription starter packs'],
        'campaigns': ['Introductory discount codes', 'Affordable bundle promotions',
                      'Win-back reactivation emails', 'Season-end clearance alerts'],
        'channels': ['Email (mass)', 'Social media organic', 'Push notifications'],
        'priority': 'MEDIUM — Volume & conversion focus'
    },
}

for seg, info in RECOMMENDATIONS.items():
    count = (df['Segment'] == seg).sum()
    pct = count / len(df) * 100
    print(f"{'='*65}")
    print(f"  {info['emoji']}  {seg.upper()}  ({count:,} customers | {pct:.1f}%)")
    print(f"  Priority: {info['priority']}")
    print(f"  Recommended Products:")
    for p in info['products']:
        print(f"    • {p}")
    print(f"  Marketing Campaigns:")
    for c in info['campaigns']:
        print(f"    → {c}")
    print(f"  Best Channels: {', '.join(info['channels'])}")
print('='*65)

## 13. Business Insights Report

In [ ]:
seg_stats = df.groupby('Segment').agg(
    Count=('Segment', 'count'),
    Avg_Income=('Income', 'mean'),
    Avg_Spending=('TotalSpending', 'mean'),
    Avg_Age=('Age', 'mean'),
    Avg_Purchases=('TotalPurchases', 'mean'),
    Total_Revenue_Proxy=('TotalSpending', 'sum')
).sort_values('Avg_Spending', ascending=False)

total_revenue = seg_stats['Total_Revenue_Proxy'].sum()
seg_stats['Revenue_Share'] = (seg_stats['Total_Revenue_Proxy'] / total_revenue * 100).round(1)
seg_stats['Customer_Share'] = (seg_stats['Count'] / len(df) * 100).round(1)

print("=" * 70)
print("  BUSINESS INSIGHTS REPORT — CUSTOMER SEGMENTATION ANALYSIS")
print("=" * 70)
print(f"\n  Total Customers Analysed : {len(df):,}")
print(f"  Number of Segments       : {df['Segment'].nunique()}")
print(f"  Features Used            : {', '.join(features)}")
print(f"  Algorithm                : K-Means++ (K={OPTIMAL_K})")

print("\n" + "-" * 70)
print("  SEGMENT PERFORMANCE OVERVIEW")
print("-" * 70)
for seg, row in seg_stats.iterrows():
    print(f"\n  [{seg}]")
    print(f"    Customers   : {row['Count']:,} ({row['Customer_Share']:.1f}% of base)")
    print(f"    Revenue Share: {row['Revenue_Share']:.1f}% of total spend")
    print(f"    Avg Income  : ${row['Avg_Income']:,.0f}")
    print(f"    Avg Spending: ${row['Avg_Spending']:,.0f}")
    print(f"    Avg Age     : {row['Avg_Age']:.0f} years")
    print(f"    Avg Purchases/period: {row['Avg_Purchases']:.1f}")

top_revenue_seg = seg_stats['Revenue_Share'].idxmax()
top_count_seg = seg_stats['Customer_Share'].idxmax()

print("\n" + "-" * 70)
print("  KEY STRATEGIC INSIGHTS")
print("-" * 70)
print(f"\n  1. Revenue Champion:")
tr = seg_stats.loc[top_revenue_seg]
print(f"     '{top_revenue_seg}' represents {tr['Customer_Share']:.1f}% of customers")
print(f"     but drives {tr['Revenue_Share']:.1f}% of total spending.")
print(f"     → Prioritise retention; margin is highest here.\n")
print(f"  2. Largest Segment:")
tc = seg_stats.loc[top_count_seg]
print(f"     '{top_count_seg}' holds {tc['Customer_Share']:.1f}% of the base.")
print(f"     Average spending = ${tc['Avg_Spending']:,.0f} per customer.")
print(f"     → Volume-based campaigns will have widest reach here.\n")
pot_idx = [s for s in seg_stats.index if 'Potential' in s]
if pot_idx:
    pot = seg_stats.loc[pot_idx[0]]
    gap = pot['Avg_Income'] - pot['Avg_Spending']
    print(f"  3. Untapped Revenue Gap:")
    print(f"     '{pot_idx[0]}' earns ${pot['Avg_Income']:,.0f} on average")
    print(f"     but spends only ${pot['Avg_Spending']:,.0f} — a ${gap:,.0f} gap.")
    print(f"     → Personalised incentives could unlock significant uplift.\n")
print("  4. Marketing Budget Allocation (recommended):")
budget_map = {
    'Luxury Shoppers': 35,
    'Potential Customers': 30,
    'Value Seekers': 20,
    'Budget Customers': 15,
}
for seg in seg_stats.index:
    alloc = budget_map.get(seg, 15)
    bar = '█' * (alloc // 2)
    print(f"     {seg:25s}: {alloc:2d}%  {bar}")
print("\n" + "=" * 70)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('Business Insights Dashboard', fontweight='bold', fontsize=16)

segs = list(seg_stats.index)
seg_bar_colors = [SEGMENT_COLORS.get(s, 'grey') for s in segs]

bars1 = axes[0].bar(segs, seg_stats.loc[segs, 'Customer_Share'],
                    color=seg_bar_colors, alpha=0.85, edgecolor='white', linewidth=0.8)
axes[0].set_title('Customer Share (%)', fontweight='bold')
axes[0].set_ylabel('%')
axes[0].tick_params(axis='x', rotation=20)
for bar in bars1:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

bars2 = axes[1].bar(segs, seg_stats.loc[segs, 'Revenue_Share'],
                    color=seg_bar_colors, alpha=0.85, edgecolor='white', linewidth=0.8)
axes[1].set_title('Revenue Share (%)', fontweight='bold')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=20)
for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.4,
                 f'{bar.get_height():.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

avg_inc = seg_stats.loc[segs, 'Avg_Income'].values
avg_spe = seg_stats.loc[segs, 'Avg_Spending'].values
x = np.arange(len(segs))
w = 0.4
axes[2].bar(x - w/2, avg_inc, w, label='Avg Income', color='#4C72B0', alpha=0.85, edgecolor='white')
axes[2].bar(x + w/2, avg_spe, w, label='Avg Spending', color='#DD8452', alpha=0.85, edgecolor='white')
axes[2].set_title('Avg Income vs Avg Spending ($)', fontweight='bold')
axes[2].set_ylabel('$')
axes[2].set_xticks(x)
axes[2].set_xticklabels(segs, rotation=20, ha='right')
axes[2].legend()
axes[2].yaxis.set_major_formatter(matplotlib.ticker.FuncFormatter(lambda y, _: f'${y/1000:.0f}K'))

plt.tight_layout()
plt.savefig('business_insights_dashboard.png', bbox_inches='tight')
plt.show()

---
## Summary

This notebook demonstrates a complete end-to-end **Customer Segmentation** pipeline:

| Step | Technique | Outcome |
|------|-----------|---------|
| Preprocessing | StandardScaler + MinMaxScaler | Clean, scaled feature matrix |
| Optimal K | Elbow Method (WCSS) | Identified inflection point |
| Cluster Quality | Silhouette Score | Validated separation |
| Clustering | K-Means++ | 4 distinct customer groups |
| Dimensionality Reduction | PCA (2D) | Visual cluster separation |
| Business Mapping | Rule-based profiling | Named segments with strategy |
| Recommendations | Segment-specific action plans | Actionable marketing playbook |

**Model saved — ready for deployment or further pipeline integration.**
